Set Evalhub environment variables

In [ ]:
EVALHUB_URL='https://evalhub-evalhub.apps.cluster-abcd.abcd.sandboxabcd.com'
TENANT_NS='evalhub'
TOKEN='sha256~abcd'
!echo $EVALHUB_URL
!echo $TENANT_NS
!echo $TOKEN

Test and check the evalhub url to make sure it is running and healthy

In [ ]:
!curl $EVALHUB_URL/api/v1/health | jq .

Check to see what providers are enabled

In [ ]:
!curl -s -H "Authorization: Bearer $TOKEN" -H "X-Tenant: $TENANT_NS" $EVALHUB_URL/api/v1/evaluations/providers | jq .

Install Python eval-hub-sdk client

In [ ]:
pip install "eval-hub-sdk[client]"

In [ ]:
pip list | grep -i eval-hub

eval-hub-sdk            1.0.5
Note: you may need to restart the kernel to use updated packages.


Set up connection to evalhub on openshift cluster

In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url="https://evalhub-evalhub.apps.cluster-abcd.abcd.sandboxabcd.com",
    tenant="evalhub",
    auth_token="sha256~abcd"
)

List evalhub providers and Garak benchmarks

In [ ]:
for provider in client.providers.list():
    print(f"{provider.resource.id}: {provider.name}")

benchmarks = client.benchmarks.list(provider_id="garak")
for b in benchmarks:
    print(f"  {b.id}: {b.name}, pass criteria: {b.pass_criteria}, primary score: {b}")

Run Garak:quick Benchmark. 
Make sure to update Model URL, model name.
ExperimentConfig is for MLFlow tracking. Make sure to update these so Mlflow is invoked

In [ ]:
from evalhub import CollectionRef
from evalhub.models import JobSubmissionRequest, ModelConfig, BenchmarkConfig, PassCriteria, PrimaryScore, ExperimentConfig, ExperimentTag

job = client.jobs.submit(JobSubmissionRequest(
    name="my-eval",
    model=ModelConfig(
        url="https://redhatai-qwen3-8b-fp8-dynamic-evalhub.apps.cluster-abcd.abcd.sandboxabcd.com/v1",
        name="redhatai-qwen3-8b-fp8-dynamic"
    ),
    experiment=ExperimentConfig(name="qwen3-quick-eval", tags=[ExperimentTag(key="model_name", value="qwen3-8b-fp8-dynamic")]),
    benchmarks=[
        BenchmarkConfig(provider_id="garak", id="quick",
            primary_score=PrimaryScore(metric="attack_success_rate", lower_is_better=True),
            pass_criteria=PassCriteria(threshold=0.3)),
    ],
), tenant="evalhub")

job_id = job.resource.id
print(f"Job ID: {job_id}")

Run GuideLLM:quick_perf_test Benchmark. 
Make sure to update Model URL, model name.
ExperimentConfig is for MLFlow tracking. Make sure to update these so Mlflow is invoked

In [ ]:
from evalhub import CollectionRef
from evalhub.models import JobSubmissionRequest, ModelConfig, BenchmarkConfig, PassCriteria, PrimaryScore, ExperimentConfig, ExperimentTag

job = client.jobs.submit(JobSubmissionRequest(
    name="my-eval",
    model=ModelConfig(
        url="https://redhatai-qwen3-8b-fp8-dynamic-evalhub.apps.cluster-abcd.abcd.sandboxabcd.com/v1",
        name="redhatai-qwen3-8b-fp8-dynamic"
    ),
    experiment=ExperimentConfig(name="qwen3-quick-perf-test", tags=[ExperimentTag(key="model_name", value="qwen3-8b-fp8-dynamic")]),
    benchmarks=[
        BenchmarkConfig(provider_id="guidellm", id="quick_perf_test",
            primary_score=PrimaryScore(metric="requests_per_second", lower_is_better=False),
            pass_criteria=PassCriteria(threshold=0.5),
            parameters={"data": "prompt_tokens=256,output_tokens=64", "max_seconds": 25, "request_type": "chat_completions", "warmup": "10%"}
        )
    ],
), tenant="evalhub")

job_id = job.resource.id
print(f"Job ID: {job_id}")

View pod logs while benchmark is running

In [ ]:
from evalhub import (
    JobLogOptions,
    SyncEvalHubClient,
    JobLogOptions,
)

with SyncEvalHubClient(base_url="https://evalhub-evalhub.apps.cluster-abcd.abcd.sandboxabcd.com",
    tenant="evalhub",
    auth_token="sha256~abcd") as client:
    for update in client.jobs.watch_logs(
        job_id,
        options=JobLogOptions(tail_lines=500),
        poll_interval=2.0,
    ):
        if update.logs:
            print(update.logs, end="")

List completed jobs

In [ ]:
from evalhub.models import JobStatus

completed_jobs = client.jobs.list(status=JobStatus.COMPLETED, limit=10)
for job in completed_jobs:
    print(f"{job.id}: {job.state}")

View Benchmark results. Update <job_id>

In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url="https://evalhub-evalhub.apps.cluster-abcd.abcd.sandboxabcd.com",
    tenant="evalhub",
    auth_token="sha256~abcd"
)
job = client.jobs.get("<job_id>")
job_id = job
print(job_id.resource.id)
print(job_id.model_dump_json(indent=2))

# Access results
if job.results:
    for benchmark in job.results.benchmarks:
        print(f"Benchmark: {benchmark.provider_id}:{benchmark.id}")
else:
    print("Job results are not yet available.") 



Test results from benchmark are not in above result. Curl evalhub API endpoint to see test results pass/fail

In [ ]:
!curl -s -H "Authorization: Bearer $TOKEN" -H "X-Tenant: evalhub" \
    $EVALHUB_URL/api/v1/evaluations/jobs/<job_id> | jq .results